# zero-day-timeline

## Overview:
What it is: A system that models the 0day lifecycle from 
CVE publication to proof-of-concept release to active exploitation in the wild.

Method: Train a regression/survival model using NVD metadata, CISA's Known Exploited Vulnerabilities catalog, Exploit-DB, and security Twitter archives. 

Output: "given this CVE's characteristics, when and how likely is active exploitation?"

Why it fits you: This speaks directly to your fascination with zero-days. It quantifies the secrecy window. 

Rich NLP territory: CVE description text, CVSS narrative, vendor advisory language, forum chatter signals. You'd be combining your data engineering background (W205) with NLP feature extraction and the fine-tuning techniques from the second paper.

Stack: Survival analysis or gradient boosting on structured features + BERT embeddings of CVE text.

Data: NVD API (free), CISA KEV (public), Exploit-DB (public).

Value signal: Directly actionable for patch prioritization. The goal is a model that communicates, "this CVE will be exploited within 7 days."

# 00: Intro and Vocab

- **CVE**: Common Vulnerabilities and Exposures
- **CVSS**: Common Vulnerability Scoring System
- **CISA**: Cybersecurity and Infrastructure Security Agency
- **KEV**: Known Exploited Vulnerability
- **NVD**: National Vulnerability Database
- **EPSS**: Exploit Prediction Scoring System
     > *Exploit Prediction Scoring System (EPSS) is a data-driven framework managed by FIRST that estimates the probability of a specific vulnerability (CVE) being exploited in the wild within the next 30 days. It helps security teams prioritize patching by focusing on active threats rather than just theoretical severity. (FIRST)*


# 01: Data Collection

Builds the master CVE table by joining:
- **CISA KEV**: confirmed exploitation dates (~1,200 CVEs)
- **NVD API**: CVSS scores, CWE, publish dates for KEV CVEs + a random unexploited sample (censored)
- **EPSS**: exploit probability scores at current date (proxy for disclosure-time signal)

Output: `data/processed/cve_master.csv`

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from collect_kev import fetch_kev
from collect_nvd import fetch_cve_batch
from collect_epss import fetch_epss_current

## Step 1: Fetch CISA KEV

In [3]:
kev = fetch_kev()
print(f"KEV entries: {len(kev)}")
kev.head()

KEV entries: 1568


,cve_id,kev_date_added,vuln_name,description
0,CVE-2009-0238,2026-04-14,Microsoft Office Remote Code Execution,Microsoft Office Excel contains a remote code ...
1,CVE-2026-32201,2026-04-14,Microsoft SharePoint Server Improper Input Val...,Microsoft SharePoint Server contains an improp...
2,CVE-2012-1854,2026-04-13,Microsoft Visual Basic for Applications Insecu...,Microsoft Visual Basic for Applications (VBA) ...
3,CVE-2025-60710,2026-04-13,Microsoft Windows Link Following Vulnerability,Microsoft Windows contains a link following vu...
4,CVE-2023-21529,2026-04-13,Microsoft Exchange Server Deserialization of U...,Microsoft Exchange Server contains a deseriali...


## Step 2: Fetch NVD metadata for KEV CVEs

**Note on rate limits:** Without an API key this takes ~2 hours for all ~1,200 KEV entries (6s/request). 
Options:
- Get a free NVD API key at https://nvd.nist.gov/developers/request-an-api-key (set `API_KEY` below, reduces wait to ~12 min)
- Use `sample_n` to test with a small subset first

In [5]:
API_KEY = '147381f4-7930-434f-bd00-4c85679aed1e'  # Replace with your NVD API key string, or leave None
DELAY = 0.6 if API_KEY else 6.0

# Test with 20 CVEs first; set sample_n=None to run all
sample_n = None

kev_ids = kev['cve_id'].tolist()
if sample_n:
    kev_ids = kev_ids[:sample_n]

nvd_kev = fetch_cve_batch(kev_ids, api_key=API_KEY, delay=DELAY)
print(f"Fetched NVD data for {len(nvd_kev)} CVEs")
nvd_kev.head()

Fetching NVD:  42%|██████████████████████████▋                                     | 653/1568 [15:31<1:40:02,  6.56s/it]

  Error fetching CVE-2023-2033: HTTPSConnectionPool(host='services.nvd.nist.gov', port=443): Max retries exceeded with url: /rest/json/cves/2.0?cveId=CVE-2023-2033 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)')))


Fetching NVD:  86%|███████████████████████████████████████████████████████▊         | 1346/1568 [34:26<06:32,  1.77s/it]

  Error fetching CVE-2019-13608: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


Fetching NVD: 100%|█████████████████████████████████████████████████████████████████| 1568/1568 [39:23<00:00,  1.51s/it]

Fetched NVD data for 1566 CVEs


,cve_id,nvd_published,cvss_score,attack_vector,attack_complexity,privileges_required,user_interaction,scope,cvss_severity,cwe
0,CVE-2009-0238,2009-02-25 16:30:00.343,8.8,NETWORK,LOW,NONE,REQUIRED,UNCHANGED,HIGH,CWE-94
1,CVE-2026-32201,2026-04-14 18:17:27.160,6.5,NETWORK,LOW,NONE,NONE,UNCHANGED,MEDIUM,CWE-20
2,CVE-2012-1854,2012-07-10 21:55:05.587,7.8,LOCAL,LOW,NONE,REQUIRED,UNCHANGED,HIGH,NVD-CWE-Other
3,CVE-2025-60710,2025-11-11 18:15:39.073,7.8,LOCAL,LOW,LOW,NONE,UNCHANGED,HIGH,CWE-59
4,CVE-2023-21529,2023-02-14 20:15:11.743,8.8,NETWORK,LOW,LOW,NONE,UNCHANGED,HIGH,CWE-502


## Step 3: Fetch EPSS scores

In [6]:
epss = fetch_epss_current(kev_ids)
print(f"EPSS scores fetched: {len(epss)}")
epss.head()

EPSS scores fetched: 1567


,cve_id,epss_score,epss_percentile
0,CVE-2026-5281,0.03278,0.87181
1,CVE-2026-3910,0.00665,0.71219
2,CVE-2026-3909,0.00288,0.52275
3,CVE-2026-35616,0.25255,0.96195
4,CVE-2026-3502,0.01481,0.80998


## Step 4: Join and compute outcome variable

In [ ]:
df = nvd_kev.merge(kev[['cve_id', 'kev_date_added']], on='cve_id', how='left')
df = df.merge(epss, on='cve_id', how='left')

# Outcome: days from NVD publish to confirmed exploitation
df['days_to_exploit'] = (df['kev_date_added'] - df['nvd_published']).dt.days
df['exploited'] = df['kev_date_added'].notna().astype(int)

# Drop rows where days_to_exploit is negative (NVD publish after KEV : data quality issue)
print(f"Negative days_to_exploit: {(df['days_to_exploit'] < 0).sum()}")
df = df[df['days_to_exploit'].isna() | (df['days_to_exploit'] >= 0)]

print(f"\nFinal dataset: {len(df)} CVEs")
print(f"Exploited: {df['exploited'].sum()} | Unexploited/censored: {(df['exploited']==0).sum()}")
df[['cve_id', 'nvd_published', 'kev_date_added', 'days_to_exploit', 'exploited', 'cvss_score']].head(10)

Negative days_to_exploit: 195

Final dataset: 1371 CVEs
Exploited: 1371 | Unexploited/censored: 0


,cve_id,nvd_published,kev_date_added,days_to_exploit,exploited,cvss_score
0,CVE-2009-0238,2009-02-25 16:30:00.343,2026-04-14,6256,1,8.8
2,CVE-2012-1854,2012-07-10 21:55:05.587,2026-04-13,5024,1,7.8
3,CVE-2025-60710,2025-11-11 18:15:39.073,2026-04-13,152,1,7.8
4,CVE-2023-21529,2023-02-14 20:15:11.743,2026-04-13,1153,1,8.8
5,CVE-2023-36424,2023-11-14 18:15:45.990,2026-04-13,880,1,7.8
6,CVE-2020-9715,2020-08-19 14:15:13.407,2026-04-13,2062,1,7.8
7,CVE-2026-21643,2026-02-06 09:15:49.330,2026-04-13,65,1,9.8
8,CVE-2026-34621,2026-04-11 07:16:03.633,2026-04-13,1,1,8.6
9,CVE-2026-1340,2026-01-29 22:15:53.313,2026-04-08,68,1,9.8
10,CVE-2026-35616,2026-04-04 01:16:39.720,2026-04-06,1,1,9.8


## Step 5: Save

In [8]:
df.to_csv('../data/processed/cve_master.csv', index=False)
print("Saved to data/processed/cve_master.csv")
df.dtypes

Saved to data/processed/cve_master.csv


cve_id                         object
nvd_published          datetime64[ns]
cvss_score                    float64
attack_vector                  object
attack_complexity              object
privileges_required            object
user_interaction               object
scope                          object
cvss_severity                  object
cwe                            object
kev_date_added         datetime64[ns]
epss_score                    float64
epss_percentile               float64
days_to_exploit                 int64
exploited                       int64
dtype: object